# Get the data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huizhml/global-vegetation-structure/blob/master/notebooks/01_get_the_data.ipynb)

The Vertical Vegetation Structure dataset gives 101 relative height (RH) metrics, RH0 to RH100, for every 10 m pixel (2020), as Cloud-Optimized GeoTIFFs on [Source Cooperative](https://source.coop/geoai-ucph/gvsm). This notebook shows three ways in:

1. **Read a window** of a 10 m tile without downloading the file.
2. **Download a whole tile.**
3. **Read the ~1 km global overview.**

How the files are laid out:

| | Path | Grid |
|---|---|---|
| 10 m | `{STORE}/2020/{TILE}/RH{n}_Q1.tif` | one Sentinel-2 (MGRS) tile, 10980 × 10980 px, in the tile's UTM zone |
| ~1 km | `{STORE}/mosaics/2020/RH{n}_Q1.tif` | the world, 0.01° (~1.1 km at the equator), EPSG:4326 |

Values are 16-bit integers in **decimetres** (divide by 10 for metres); `32767` is nodata. `Q1` is the median prediction.

In [ ]:
%pip install -q rasterio mgrs

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.warp import transform
from rasterio.windows import from_bounds

# Reading over HTTP: skip directory listings, merge neighbouring byte ranges.
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["GDAL_HTTP_MERGE_CONSECUTIVE_RANGES"] = "YES"

STORE = "https://data.source.coop/geoai-ucph/gvsm"
YEAR = 2020
NODATA = 32767


def tile_url(tile, rh, year=YEAR):
    """10 m file: one per Sentinel-2 (MGRS) tile and RH. Q1 = the median prediction."""
    return f"{STORE}/{year}/{tile}/RH{rh}_Q1.tif"


def mosaic_url(rh, year=YEAR):
    """~1 km global overview: one file per RH."""
    return f"{STORE}/mosaics/{year}/RH{rh}_Q1.tif"


def read_box(url, lon, lat, half_m):
    """Read a square of 2*half_m metres around (lon, lat) from a 10 m tile, in metres.

    Only the bytes of that window are fetched: the files are Cloud-Optimized GeoTIFFs.
    Values are stored as decimetres with 32767 as nodata.
    """
    with rasterio.open(url) as src:
        xs, ys = transform("EPSG:4326", src.crs, [lon], [lat])
        x, y = xs[0], ys[0]
        win = from_bounds(x - half_m, y - half_m, x + half_m, y + half_m, src.transform)
        data = src.read(1, window=win.round_offsets().round_lengths()).astype("float32")
    data[data == NODATA] = np.nan
    return data / 10

## Which tile?

The 10 m data is split along the Sentinel-2 tiling grid, whose tiles are named after the 100 km squares of the MGRS grid (e.g. `32UNA`). The `mgrs` package gives the square for a point:

In [ ]:
import mgrs

# A forest in the Spessart, Germany — the one the explorer's guided tour visits.
lon, lat = 9.312, 49.937
tile = mgrs.MGRS().toMGRS(lat, lon, MGRSPrecision=0)
tile

## 1. Read a window

A 3 km × 3 km window around the point, for RH98 (≈ canopy top height) and RH25 (where a quarter of the returned lidar energy has accumulated). Only the bytes of the window are fetched.

In [ ]:
half = 1500  # metres either side of the point
rh98 = read_box(tile_url(tile, 98), lon, lat, half)
rh25 = read_box(tile_url(tile, 25), lon, lat, half)
print(rh98.shape, "pixels of 10 m")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, img, name, vmax in [(axes[0], rh98, "RH98", 50), (axes[1], rh25, "RH25", 12)]:
    im = ax.imshow(img, cmap="inferno", vmin=0, vmax=vmax)
    ax.set_title(f"{name} (m)")
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

## 2. Download a whole tile

One file per tile and RH, roughly 100–170 MB each. The name below follows the explorer's download panel, which keeps product, year, tile and RH in the name.

In [ ]:
import requests

rh = 98
out = f"vsm_10m_{tile}_{YEAR}_RH{rh}_Q1.tif"
with requests.get(tile_url(tile, rh), stream=True, timeout=60) as r:
    r.raise_for_status()
    with open(out, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
print(out, round(os.path.getsize(out) / 1e6), "MB")

## 3. The ~1 km global overview

One file per RH covers the world. Reading it at 1/16 of its resolution uses the file's internal overviews, so only a few MB come over the network.

In [ ]:
from rasterio.enums import Resampling

with rasterio.open(mosaic_url(98)) as src:
    world = src.read(1, out_shape=(src.height // 16, src.width // 16), resampling=Resampling.nearest)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
world = world.astype("float32")
world[world == NODATA] = np.nan
world /= 10

plt.figure(figsize=(12, 5))
plt.imshow(world, cmap="inferno", vmin=0, vmax=50, extent=extent)
plt.colorbar(label="RH98 (m)", shrink=0.8)
plt.title("RH98, ~1 km global overview (read at 1/16 resolution)")
plt.show()

## More

- Many tiles or RHs at once: the explorer's **Download data** panel writes a shell script that fetches them with `curl`, names each file and resumes broken transfers.
- For bulk access to the full dataset, contact huzh@di.ku.dk.
- Licence: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Please cite: Zhang, H., Lang, N., Mazurczyk, M., Oehmcke, S., Huang, K., Brandt, M., Fensholt, R., Kariryaa, A., & Igel, C. (2026). A vertical vegetation structure model of the Earth. *EarthArXiv preprint*. [doi:10.31223/X5M22F](https://doi.org/10.31223/X5M22F)